In [15]:
! pip3 install requests


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import requests
import json
import os

True

In [32]:
API_KEY = "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"
MAX_CHUNK_SIZE = 3000  # Максимальный размер чанка

In [26]:
def llm_request(prompt: str, api_key: str) -> str:
  try:
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        data=json.dumps({
            "model": "mistralai/mistral-small-3.2-24b-instruct:free",
            "messages": [{"role": "user", "content": prompt}],
        })
    )

    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content']
    else:
        return f"Ошибка {response.status_code}: {response.text}"

  except Exception as e:
    return f"Ошибка при запросе: {str(e)}"

In [46]:
! pip install sentence-transformers nltk razdel


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from razdel import sentenize
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
import numpy as np
import os
import json

# Инициализация модели и токенизатора XLM-RoBERTa
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_sentences(text: str):
    """Токенизация текста на предложения с помощью razdel"""
    return [sentence.text for sentence in sentenize(text)]

def get_embeddings(sentences):
    """Получение эмбеддингов предложений через XLM-RoBERTa"""
    inputs = tokenizer(sentences, padding=True, truncation=True, max_length=512, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Mean pooling
    last_hidden_states = outputs.last_hidden_state
    attention_mask = inputs['attention_mask']
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_states.size()).float()
    sum_embeddings = torch.sum(last_hidden_states * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return (sum_embeddings / sum_mask).numpy()

def semantic_chunking_xlm(text: str, 
                         source: str = "local_text", 
                         window_size: int = 3, 
                         similarity_threshold: float = 0.50):
    """
    Семантическое разбиение текста на чанки с использованием:
    - razdel для токенизации предложений
    - xlm-roberta-base для эмбеддингов
    """
    # 1. Разбиваем текст на предложения
    sentences = get_sentences(text)
    if len(sentences) == 0:
        return

    # 2. Получаем эмбеддинги
    embeddings = get_embeddings(sentences)

    # 3. Вычисляем матрицу косинусной схожести
    sim_matrix = F.cosine_similarity(
        torch.from_numpy(embeddings).unsqueeze(1),
        torch.from_numpy(embeddings).unsqueeze(0),
        dim=2
    ).numpy()

    # 4. Группируем предложения в чанки
    chunks = []
    current_chunk = []
    
    for i in range(len(sentences)):
        if not current_chunk:
            current_chunk.append(sentences[i])
        else:
            # Вычисляем среднюю схожесть с последними N предложениями
            last_indices = range(max(0, len(current_chunk) - window_size), len(current_chunk))
            avg_similarity = np.mean([sim_matrix[i][j] for j in last_indices])
            
            if avg_similarity >= similarity_threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]
    
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    # 5. Сохраняем результаты
    os.makedirs('./chunks', exist_ok=True)
    for i, chunk in enumerate(chunks, 1):
        chunk_data = {
            "id": f"chunk_{i:02d}",
            "source": source,
            "content": chunk,
            "size": len(chunk),
            "num_sentences": chunk.count('. ') + 1  # Простая оценка кол-ва предложений
        }
        with open(f"./chunks/chunk_{i:02d}.json", "w", encoding="utf-8") as f:
            json.dump(chunk_data, f, ensure_ascii=False, indent=2)

In [20]:
! pip3 install PyPDF2


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import PyPDF2
def read_text_from_pdf(pdf_file_path: str):
    if not os.path.exists(pdf_file_path) or not pdf_file_path.lower().endswith(".pdf"):
        return ""
    with open(pdf_file_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ''.join(page.extract_text() for page in reader.pages)  # Более лаконичное извлечение текста
    return text

In [65]:
os.makedirs('./chunks', exist_ok=True)
text = read_text_from_pdf('text.pdf')

In [31]:
with open('./res.txt', 'w', encoding="utf-8") as f:
    f.write(text)

In [77]:
semantic_chunking_xlm(text)